# Card sorting sobre PRs aceptados después de retrabajo

## Pregunta de investigación

**¿Qué tipos de problemas hacen que los pull requests generados por agentes de IA no sean aceptados inmediatamente y requieran retrabajo antes de integrarse?**

En este notebook documentamos el flujo que usamos para pasar desde el universo completo de PRs del dataset AIDev hasta una muestra revisable mediante card sorting abierto. Nos concentramos en `merged_after_rework`: PRs que finalmente fueron mergeados, pero solo después de commits adicionales y comentarios humanos observables.

## Problema, motivación y consecuencias

Medir solo si un PR fue mergeado no nos permite explicar qué ocurrió durante la revisión. Un PR puede terminar aceptado y aun así haber requerido correcciones, aclaraciones o ajustes sustantivos antes del merge.

Por eso observamos la zona intermedia: contribuciones de agentes de IA que no fueron aceptadas inmediatamente, pero que sí lograron integrarse después de intervención humana y commits adicionales.

## Enfoque metodológico según Zimmermann

Adaptamos la referencia `docs/card-sorting.pdf`, que organiza el card sorting en tres fases: **Preparation**, **Execution** y **Analysis**. Mantenemos el enfoque **abierto** para que las categorías emerjan desde las tarjetas, en lugar de imponer una taxonomía previa.

### 1) Preparation

Construimos la población operacional, declaramos criterios de inclusión/exclusión, calculamos pérdidas del embudo, estratificamos por agente y generamos una tarjeta por PR con identificador, contexto y evidencia textual humana.

### 2) Execution

Clasificamos manualmente las tarjetas en grupos con títulos descriptivos. Si una tarjeta es ambigua, la separamos para revisión; si no responde a la pregunta, la marcamos como descartable.

### 3) Analysis

Revisamos consistencia, consolidamos grupos similares en una taxonomía jerárquica y cruzamos las categorías con métricas de agente, lenguaje, tipo de tarea, comentarios humanos y tiempo hasta aceptación.

## Embudo de datos

Calculamos el embudo desde el resumen de muestreo generado por `sampling/stratified_sampler.py`. Además del total por paso, reportamos retención y pérdida porcentual contra el universo bruto AIDev, según los comentarios metodológicos recibidos.

## Universo bruto antes de construir la población operacional

La tabla siguiente muestra los cortes principales antes de construir la muestra. La pérdida porcentual queda explicitada en la tabla de embudo inmediatamente posterior.

## Archivos usados

Las rutas vienen desde las constantes de los scripts de muestreo y preparacion. El notebook no mantiene rutas duplicadas salvo la busqueda minima de la raiz del repositorio.

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start=Path.cwd()):
    for path in [start.resolve(), *start.resolve().parents]:
        if (path / "exploration" / "aidev").exists():
            return path
    raise RuntimeError("No se encontro la raiz del repositorio")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from exploration.aidev.notebook_flow import (
    build_agent_distribution,
    build_evidence_tables,
    build_files_table,
    build_funnel,
    build_outputs_flow,
    build_preparation_flow,
    build_raw_overview,
    build_template_preview,
    load_flow_artifacts,
    validate_flow,
)
from exploration.aidev.preparation.rejection_cards import MANUAL_TEMPLATE_FIELDS
from exploration.aidev.sampling.stratified_sampler import POPULATION_MODE, STRATA_FIELDS

artifacts = load_flow_artifacts(ROOT)
sampling_summary = artifacts.sampling_summary
preparation_summary = artifacts.preparation_summary
filter_counts = sampling_summary["population_filter_counts"]
sample_df = artifacts.sample_df
cards_df = artifacts.cards_df
template_df = artifacts.template_df

POPULATION_MODE, STRATA_FIELDS, MANUAL_TEMPLATE_FIELDS[-1], len(sample_df), len(cards_df)


('merged-after-rework', ['agent'], 'categoria_retrabajo_pre_merge', 300, 300)

## Paso 0: filtros poblacionales antes de estratificar

El corte clave que usamos es `merged_at` no nulo, `commit_count > 1` y `human_comment_count > 0`. Esta decisión evita mezclar rechazos definitivos con aceptaciones después de retrabajo y mejora la trazabilidad entre evidencia textual humana y categoría asignada.

In [2]:
build_files_table(artifacts)


,artefacto,ruta
0,Resumen de filtros poblacionales,exploration/aidev/sampling/outputs/merged_afte...
1,Resumen de muestreo,exploration/aidev/sampling/outputs/merged_afte...
2,Poblacion operacional,exploration/aidev/sampling/outputs/merged_afte...
3,Muestra estratificada,exploration/aidev/sampling/outputs/merged_afte...
4,Resumen de preparacion,exploration/aidev/preparation/outputs/merged_a...
5,Tarjetas con evidencia,exploration/aidev/preparation/outputs/merged_a...
6,Plantilla manual,exploration/aidev/preparation/outputs/merged_a...


In [3]:
build_raw_overview(sampling_summary)


,metrica,definicion,total,porcentaje_del_universo,perdida_acumulada
0,PRs totales en pull_request,Todos los registros del parquet pull_request,33596,1.000000,0.000000
1,PRs cerrados,state = closed,31284,0.931182,0.068818
2,PRs mergeados,merged_at no nulo,24014,0.714787,0.285213
3,PRs cerrados sin merge,state = closed y merged_at nulo,7270,0.216395,0.783605
4,PRs mergeados con commits adicionales,merged_at no nulo y commit_count > 1,6884,0.204905,0.795095
5,Poblacion operacional,"merged_at no nulo, commit_count > 1 y human_co...",3166,0.094237,0.905763


## Criterios de inclusión/exclusión y pérdidas porcentuales

Incluimos solo PRs que pertenecen a AIDev, están cerrados y mergeados, tienen `commit_count > 1`, tienen `human_comment_count > 0` y corresponden al caso operacional `merged_after_rework`. Excluimos PRs abiertos, cerrados sin merge, mergeados sin commits adicionales, sin comentarios humanos observables y casos `rejected`, porque aquí estudiamos retrabajo pre-merge y no rechazo definitivo.

La tabla de embudo se calcula dinámicamente desde `sampling_summary` en la siguiente celda, incluyendo retención y pérdida contra el universo bruto.

## Paso 1: estratificación por agente

Una vez construida la población operacional, calculamos una muestra estratificada usando `agent` como variable de estratificación. La asignación vigente es proporcional por agente:

```text
n_h = round((N_h / N) * n)
```

donde `N_h` es el tamaño del estrato, `N` la población operacional total, `n` el tamaño de muestra objetivo y `n_h` la cuota asignada al agente. Con `N = 3.166` y `n = 300`, obtenemos las cuotas que se muestran abajo.

Con corrección por población finita, 95% de confianza (`z = 1,96`) y máxima varianza (`p = 0,5`), `n = 300` implica un error aproximado de ±5,38%. Para cumplir error ≤ 5%, el plan de mejora es aumentar la muestra a `n ≈ 343` antes del card sorting final.

In [4]:
embudo = build_funnel(artifacts).copy()
embudo["perdida_vs_universo"] = 1 - embudo["retencion_vs_universo"]

criterios_embudo = {
    "Universo bruto AIDev": {
        "criterio_inclusion": "Todos los PRs registrados en AIDev/pull_request",
        "criterio_exclusion": "Ninguno en este paso",
    },
    "PRs mergeados": {
        "criterio_inclusion": "`merged_at` no nulo",
        "criterio_exclusion": "PRs abiertos o cerrados sin merge",
    },
    "PRs mergeados con commits adicionales": {
        "criterio_inclusion": "PRs mergeados con `commit_count > 1`",
        "criterio_exclusion": "PRs mergeados con un solo commit",
    },
    "Poblacion antes de estratificar": {
        "criterio_inclusion": "PRs mergeados con `commit_count > 1` y `human_comment_count > 0`",
        "criterio_exclusion": "PRs sin comentario humano observable",
    },
    "Muestra estratificada por agente": {
        "criterio_inclusion": "Selección aleatoria estratificada por `agent` con seed 20260510",
        "criterio_exclusion": "PRs de la población no seleccionados por cuota muestral",
    },
    "Tarjetas candidatas": {
        "criterio_inclusion": "Una tarjeta por PR seleccionado en la muestra",
        "criterio_exclusion": "Duplicados de `pr_id`, si existieran",
    },
    "Tarjetas listas para card sorting": {
        "criterio_inclusion": "Tarjetas con `human_comment_count > 0`",
        "criterio_exclusion": "Tarjetas sin evidencia humana mínima",
    },
    "Plantilla manual": {
        "criterio_inclusion": "Tarjetas listas exportadas para categorización manual",
        "criterio_exclusion": "Casos no trazables a una tarjeta válida",
    },
}

embudo_dinamico = embudo[["paso", "total", "retencion_vs_universo", "perdida_vs_universo"]].copy()
embudo_dinamico["criterio_inclusion"] = embudo_dinamico["paso"].map(
    lambda paso: criterios_embudo.get(paso, {}).get("criterio_inclusion", "")
)
embudo_dinamico["criterio_exclusion"] = embudo_dinamico["paso"].map(
    lambda paso: criterios_embudo.get(paso, {}).get("criterio_exclusion", "")
)
embudo_dinamico = embudo_dinamico[[
    "paso",
    "criterio_inclusion",
    "criterio_exclusion",
    "total",
    "retencion_vs_universo",
    "perdida_vs_universo",
]]
for columna in ["retencion_vs_universo", "perdida_vs_universo"]:
    embudo_dinamico[columna] = embudo_dinamico[columna].map(lambda value: f"{value:.2%}")
embudo_dinamico


,paso,criterio_inclusion,criterio_exclusion,total,retencion_vs_universo,perdida_vs_universo
0,Universo bruto AIDev,Todos los PRs registrados en AIDev/pull_request,Ninguno en este paso,33596,100.00%,0.00%
1,PRs mergeados,`merged_at` no nulo,PRs abiertos o cerrados sin merge,24014,71.48%,28.52%
2,PRs mergeados con commits adicionales,PRs mergeados con `commit_count > 1`,PRs mergeados con un solo commit,6884,20.49%,79.51%
3,Poblacion antes de estratificar,PRs mergeados con `commit_count > 1` y `human_...,PRs sin comentario humano observable,3166,9.42%,90.58%
4,Muestra estratificada por agente,Selección aleatoria estratificada por `agent` ...,PRs de la población no seleccionados por cuota...,300,0.89%,99.11%
5,Tarjetas candidatas,Una tarjeta por PR seleccionado en la muestra,"Duplicados de `pr_id`, si existieran",300,0.89%,99.11%
6,Tarjetas listas para card sorting,Tarjetas con `human_comment_count > 0`,Tarjetas sin evidencia humana mínima,300,0.89%,99.11%
7,Plantilla manual,Tarjetas listas exportadas para categorización...,Casos no trazables a una tarjeta válida,300,0.89%,99.11%


### Distribucion por agente

La distribucion compara poblacion, cuotas de muestreo y tarjetas finales.

In [5]:
build_agent_distribution(artifacts)


,poblacion_antes_de_estratificar,muestra_estratificada,tarjetas_finales
Copilot,1526,145,145
Devin,907,86,86
OpenAI_Codex,480,45,45
Cursor,180,17,17
Claude_Code,73,7,7


## Paso 2: Preparation de tarjetas

En la fase **Preparation**, convertimos cada PR de la muestra en una tarjeta. Cada tarjeta conserva un identificador (`card_id`), contexto mínimo del PR, agente, tiempos de aceptación y una cita textual humana cuando existe evidencia suficiente. El filtro `human_comment_count > 0` queda como guardia de calidad.

In [7]:
build_preparation_flow(artifacts)


,paso,criterio,filas
0,Tarjetas candidatas,PRs de la muestra antes de la guardia de calidad,300
1,Tarjetas listas,human_comment_count > 0,300
2,Descartes,sin comentarios humanos detectados en evidencia,0


### Evidencia disponible tras Preparation

La preparación prioriza reviews, comentarios inline, comentarios generales y timeline. La tabla resume la fuente principal seleccionada para cada tarjeta; esta evidencia será la base de la `cita_textual_retrabajo` usada durante el sorting.

In [8]:
evidence, review_states = build_evidence_tables(artifacts)
display(evidence)
display(review_states)


,fuente_evidencia,tarjetas
0,pr_review_comment,160
1,pr_comment,89
2,pr_review,51


,estado_review,tarjetas
0,COMMENTED,256
2,APPROVED,25
1,CHANGES_REQUESTED,18
3,DISMISSED,1


## Paso 3: Execution y Analysis del card sorting

El flujo vigente produce tarjetas con evidencia y una plantilla manual. En **Execution**, usamos una vista reducida para agrupar tarjetas y asignar categorías emergentes. En **Analysis**, revisamos consistencia, consolidamos grupos similares y cruzamos la taxonomía con métricas de esfuerzo/tiempo.

In [9]:
build_outputs_flow(artifacts)


,artefacto,ruta,filas
0,Tarjetas con evidencia,exploration/aidev/preparation/outputs/merged_a...,300
1,Plantilla manual,exploration/aidev/preparation/outputs/merged_a...,300


### Tabla reducida para categorizar

Para responder mejor la pregunta de investigación, reducimos la tabla manual a las columnas que realmente sostienen la decisión cualitativa. La definición de columnas y la vista de categorización se calculan dinámicamente en las siguientes celdas.

La categoría solo es sound si queda respaldada por la cita textual. Si la cita no existe o no responde la pregunta, marcamos la tarjeta como ambigua o descartable.

In [12]:
columnas_categorizacion = pd.DataFrame([
    {"columna": "card_id", "uso_metodologico": "identificador único para reconstruir la tarjeta"},
    {"columna": "pr_id", "uso_metodologico": "trazabilidad hacia el PR original"},
    {"columna": "agent", "uso_metodologico": "estrato/agente de origen"},
    {"columna": "html_url", "uso_metodologico": "enlace para revisar contexto si la cita no basta"},
    {"columna": "cita_textual_retrabajo", "uso_metodologico": "fragmento humano que sustenta la categoría"},
    {"columna": "evidence_source", "uso_metodologico": "fuente de la cita seleccionada"},
    {"columna": "evidence_created_at", "uso_metodologico": "fecha de la evidencia usada como cita"},
    {"columna": "merged_at", "uso_metodologico": "control temporal para distinguir evidencia pre/post merge"},
    {"columna": "categoria_retrabajo_pre_merge", "uso_metodologico": "categoría emergente asignada durante el card sorting"},
    {"columna": "justificacion_breve", "uso_metodologico": "explicación breve de por qué la cita responde la pregunta"},
])
columnas_categorizacion


,columna,uso_metodologico
0,card_id,identificador único para reconstruir la tarjeta
1,pr_id,trazabilidad hacia el PR original
2,agent,estrato/agente de origen
3,html_url,enlace para revisar contexto si la cita no basta
4,cita_textual_retrabajo,fragmento humano que sustenta la categoría
5,evidence_source,fuente de la cita seleccionada
6,evidence_created_at,fecha de la evidencia usada como cita
7,merged_at,control temporal para distinguir evidencia pre...
8,categoria_retrabajo_pre_merge,categoría emergente asignada durante el card s...
9,justificacion_breve,explicación breve de por qué la cita responde ...


In [13]:
javier_categories_path = ROOT / "exploration/aidev/preparation/outputs/merged_after_rework_manual_categories_template_Javier.csv"
tabla_reducida = pd.read_csv(javier_categories_path)
columnas_vista = [
    "card_id",
    "pr_id",
    "agent",
    "html_url",
    "cita_textual_retrabajo",
    "categoria_retrabajo_pre_merge",
    "justificacion_breve",
]
tabla_reducida[columnas_vista].head()


,card_id,pr_id,agent,html_url,cita_textual_retrabajo,categoria_retrabajo_pre_merge,justificacion_breve
0,3078006902-A,3078006902,Claude_Code,https://github.com/spacelift-io/spacectl/pull/324,z,Ausencia de Patrón de Referencia Documentado (...,Primera iteración: Claude intentó migrar TODO ...
1,3154662508-A,3154662508,Claude_Code,https://github.com/maybe-finance/maybe/pull/2389,z,Fallos Secuenciales en Validaciones Automatizadas,11 commits iniciales desencadenaron 5 checks e...
2,3184725856-A,3184725856,Claude_Code,https://github.com/oven-sh/bun/pull/20698,z,Test Flaky No Relacionado Bloqueando Merge (Yo...,Cambio en napi.cpp (NAPI_AUTO_LENGTH) fue corr...
3,3193198936-A,3193198936,Claude_Code,https://github.com/tphakala/birdnet-go/pull/841,z,Merge de Código Incompleto con Deuda Técnica A...,"PR marcada explícitamente como ""Phase 8"". Owne..."
4,3224085262-A,3224085262,Claude_Code,https://github.com/karakeep-app/karakeep/pull/...,z,Arquitectura No Validada Antes de Implementación,xuatz implementó setting en BASE DE DATOS (ALT...


## Validaciones de consistencia

Estas validaciones hacen explicitos los supuestos del flujo: poblacion `merged_after_rework`, estratificacion por agente, muestra de 300 PRs y una tarjeta final por PR.

In [14]:
validate_flow(artifacts)


'Validaciones completadas'

## Lectura metodológica y soundness

El flujo parte del universo completo de PRs, filtra antes del muestreo los casos que no muestran retrabajo observable y conserva una muestra estratificada por agente. La interpretación cualitativa debe enfocarse en motivos de retrabajo antes del merge, no en rechazo definitivo.

Para aplicar soundness, cada etiqueta manual debe cumplir cinco condiciones: (1) estar respaldada por una cita textual humana cuando exista evidencia; (2) responder directamente qué problema impidió aceptación inmediata; (3) distinguir retrabajo pre-merge de rechazo definitivo; (4) conservar trazabilidad `card_id → PR → evidencia → categoría`; y (5) no tratar la frecuencia de tarjetas como importancia absoluta, siguiendo la cautela metodológica de Zimmermann sobre cuantificar datos cualitativos.

## Plan de mejora para la presentación

La presentación debe cubrir: problema y pregunta; dataset AIDev; embudo con pérdidas porcentuales; criterios de inclusión/exclusión; tres etapas de card sorting; fórmula de estratificación; confianza/error y ajuste requerido de `n = 300` a `n ≈ 343`; tabla reducida con cita textual; soundness; y resultados esperados de taxonomía más métricas de esfuerzo/tiempo.